## Configuration

In [ ]:
import os
import cv2
import random
from pathlib import Path
from inference_sdk import InferenceHTTPClient
import supervision as sv


ROBOFLOW_API_KEY = "[API_KEY]" 
RAW_DATA_DIR = Path("") 
PROCESSED_DATA_DIR = Path("")

TRAIN_SPLIT = 1 # 80/20 for the model
EXTRA_RELATIVE_PADDING = 0.4 # padding --> according to the paper

## Dataset Distribution

In [16]:

classes = [d.name for d in RAW_DATA_DIR.iterdir() if d.is_dir()]
class_counts = {}

for chord_class in classes:
    class_dir = RAW_DATA_DIR / chord_class
    num_files = len([f for f in class_dir.iterdir() if f.is_file()])
    class_counts[chord_class] = num_files
    print(f"{chord_class}: {num_files} images")

total_images = sum(class_counts.values())
print(f"Total images in all classes: {total_images}")
if total_images > 0:
    for c in classes:
        pct = (class_counts[c] / total_images) * 100 if total_images else 0
        print(f"{c}: {pct:.2f}% of dataset")
else:
    print("WARNING: No images found in the raw dataset folders.")

C: 60 images
D: 61 images
Total images in all classes: 121
C: 49.59% of dataset
D: 50.41% of dataset


## Cropping Functions

In [17]:
def extract_and_save_fretboard(img_path, save_path):
    image = cv2.imread(str(img_path))
    if image is None:
        print(f"Error reading image: {img_path}")
        return False

    try:
     
        result = client.run_workflow(
            workspace_name="test-kpcsq",
            workflow_id="find-fretboards",
            images={"image": str(img_path)},
            )
        detections = sv.Detections.from_inference(result[0]["predictions"])
    except Exception as e:
        print(f"API Error on {img_path}: {e}")
        return False

    if len(detections) == 0:
        print(f"No fretboard detected in: {img_path}")
        return False

    x1, y1, x2, y2 = map(int, detections.xyxy[0])
    h, w = image.shape[:2]

    pad_x = int((x2 - x1) * EXTRA_RELATIVE_PADDING)
    pad_y = int((y2 - y1) * EXTRA_RELATIVE_PADDING)

    crop = image[
        max(0, y1 - pad_y) : min(h, y2 + pad_y),
        max(0, x1 - pad_x) : min(w, x2 + pad_x)
    ]

    cv2.imwrite(str(save_path), crop)
    return True

## Processing Loop

In [18]:
client = InferenceHTTPClient(
    api_url="http://localhost:9001/",
    api_key=ROBOFLOW_API_KEY,
)

for split in ['train', 'val']:
    for chord_class in classes:
        (PROCESSED_DATA_DIR / split / chord_class).mkdir(parents=True, exist_ok=True)

print(f"Found classes: {classes}")
print(f"Directory structure created at: {PROCESSED_DATA_DIR.absolute()}")

Found classes: ['C', 'D']
Directory structure created at: /Users/jasraj/Desktop/CV_Proj/dataset/fb_prod


In [19]:
total_processed = 0
total_failed = 0

for chord_class in classes:
    class_dir = RAW_DATA_DIR / chord_class
    
    images = []
    for ext in ['*.jpg', '*.jpeg', '*.png']:
        images.extend(list(class_dir.glob(ext)))
        images.extend(list(class_dir.glob(ext.upper())))
    
    random.shuffle(images)
    
    split_idx = int(len(images) * TRAIN_SPLIT)
    
    train_images = images[:split_idx]
    val_images = images[split_idx:]
    
    print(f"\nProcessing Class '{chord_class}' | Total: {len(images)} | Train: {len(train_images)} | Val: {len(val_images)}")
    counter = 500
    for i, img_path in enumerate(train_images):
        save_path = PROCESSED_DATA_DIR / 'train' / chord_class / f"{chord_class}_train_{counter}.jpg"
        counter+=1
        if extract_and_save_fretboard(img_path, save_path):
            total_processed += 1
        else:
            total_failed += 1
            
    counter = 500
    for i, img_path in enumerate(val_images):
        save_path = PROCESSED_DATA_DIR / 'val' / chord_class / f"{chord_class}_val_{counter}.jpg"
        counter+=1
        if extract_and_save_fretboard(img_path, save_path):
            total_processed += 1
        else:
            total_failed += 1

print(f"Processed and saved: {total_processed} images")
print(f"Failed or no fretboard detected: {total_failed} images")


Processing Class 'C' | Total: 60 | Train: 36 | Val: 24

Processing Class 'D' | Total: 60 | Train: 36 | Val: 24
Processed and saved: 120 images
Failed or no fretboard detected: 0 images


## Helper Functions

In [ ]:
import os
import re
from pathlib import Path

PROCESSED_DATA_DIR = Path("")
train_dir = PROCESSED_DATA_DIR / 'val' / 'Em'

pattern = re.compile(r"^(Em_val_)(\d+)(\..+)$")
counter = 122
files = sorted(train_dir.glob("*val*.jpg"))
for file in files:
    m = pattern.match(file.name)
    if m:
        new_name = f"Em_val_{counter}{m.group(3)}"
        counter+=1
        new_path = file.with_name(new_name)
        os.rename(file, new_path)
        print(f"Renamed {file.name} -> {new_name}")

Renamed Em_val_0.jpg -> Em_val_122.jpg
Renamed Em_val_1.jpg -> Em_val_123.jpg
Renamed Em_val_10.jpg -> Em_val_124.jpg
Renamed Em_val_11.jpg -> Em_val_125.jpg
Renamed Em_val_12.jpg -> Em_val_126.jpg
Renamed Em_val_13.jpg -> Em_val_127.jpg
Renamed Em_val_14.jpg -> Em_val_128.jpg
Renamed Em_val_15.jpg -> Em_val_129.jpg
Renamed Em_val_16.jpg -> Em_val_130.jpg
Renamed Em_val_17.jpg -> Em_val_131.jpg
Renamed Em_val_18.jpg -> Em_val_132.jpg
Renamed Em_val_19.jpg -> Em_val_133.jpg
Renamed Em_val_2.jpg -> Em_val_134.jpg
Renamed Em_val_20.jpg -> Em_val_135.jpg
Renamed Em_val_21.jpg -> Em_val_136.jpg
Renamed Em_val_3.jpg -> Em_val_137.jpg
Renamed Em_val_4.jpg -> Em_val_138.jpg
Renamed Em_val_5.jpg -> Em_val_139.jpg
Renamed Em_val_6.jpg -> Em_val_140.jpg
Renamed Em_val_7.jpg -> Em_val_141.jpg
Renamed Em_val_8.jpg -> Em_val_142.jpg
Renamed Em_val_9.jpg -> Em_val_143.jpg
